# Monte Carlo Pi — How Randomness Creates Order

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Notebook Guide (connected to Lab 01: The Invisible Hole in Your Eye)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/appendix-1.ipynb)

**What you will do:** throw different numbers of random darts at the square to see how the pi estimate's error shrinks -- and how unevenly -- as more random data accumulates.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Simulation

**Concept:** Averaging many random samples can converge on a precise estimate—much as the brain pools noisy neural signals into a stable perception. This simulation estimates the value of π by throwing random "darts" at a square and counting how many land inside a circle.

**Relevant Labs:** Lab 01 (prediction from sparse data), Lab 14 (why the brain trusts a strong signal more than a weak one)

In [ ]:
# Monte Carlo Pi Estimation

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

rng = np.random.default_rng(42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Generate one set of dart throws, shared by both panels ---
n_total = 5000
x = rng.uniform(-1, 1, n_total)
y = rng.uniform(-1, 1, n_total)
inside = x**2 + y**2 <= 1
final_pi_estimate = 4 * np.sum(inside) / n_total

# --- Left panel: the dartboard ---
ax1.scatter(x[inside], y[inside], c='steelblue', s=1, alpha=0.5,
            label='Inside circle')
ax1.scatter(x[~inside], y[~inside], c='salmon', s=1, alpha=0.5,
            label='Outside circle')
circle = Circle((0, 0), 1, fill=False, color='black', linewidth=2)
ax1.add_patch(circle)
ax1.set_xlim(-1.1, 1.1)
ax1.set_ylim(-1.1, 1.1)
ax1.set_aspect('equal')
ax1.set_title(f'{n_total} random darts → π ≈ {final_pi_estimate:.4f}')
ax1.legend(loc='upper right', fontsize=9)

# --- Right panel: convergence, averaged over repeated draws ---
ns = [10, 50, 100, 500, 1000, 2000, 5000]
n_repeats = 30
mean_estimates = []
std_estimates = []
for n in ns:
    estimates = []
    for _ in range(n_repeats):
        xr = rng.uniform(-1, 1, n)
        yr = rng.uniform(-1, 1, n)
        estimates.append(4 * np.sum(xr**2 + yr**2 <= 1) / n)
    mean_estimates.append(np.mean(estimates))
    std_estimates.append(np.std(estimates))

mean_estimates = np.array(mean_estimates)
std_estimates = np.array(std_estimates)

ax2.plot(ns, mean_estimates, 'o-', color='steelblue', linewidth=2,
         markersize=8, label=f'Average of {n_repeats} runs')
ax2.fill_between(ns, mean_estimates - std_estimates,
                  mean_estimates + std_estimates, color='steelblue',
                  alpha=0.2, label='±1 SD across runs')
ax2.axhline(y=np.pi, color='red', linestyle='--', linewidth=1.5,
            label=f'True π = {np.pi:.4f}')
ax2.set_xlabel('Number of darts', fontsize=12)
ax2.set_ylabel('Estimated π', fontsize=12)
ax2.set_title('More Data, On Average, Means a Better Estimate')
ax2.legend(fontsize=10)
ax2.set_xscale('log')

plt.tight_layout()
plt.savefig('monte_carlo_pi.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: averaging many random samples can converge on a "
      "precise estimate, much as the brain pools noisy signals into a "
      "stable perception.")
print(f"With {n_total} random darts, one run estimates π as "
      f"{final_pi_estimate:.4f}")
print(f"Averaged over {n_repeats} runs, the estimate at n={ns[-1]} is "
      f"{mean_estimates[-1]:.4f} (SD {std_estimates[-1]:.4f})")
print(f"The true value is {np.pi:.4f}")

## Change the parameters

**n_total** is how many random points ("darts") are thrown. **seed** changes which
particular random points are drawn, without changing how many there are. The estimate
reuses the same inside-the-circle test as the cell above.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, IntSlider

def explore(n_total=5000, seed=42):
    np.random.seed(seed)
    x = np.random.uniform(-1, 1, n_total)
    y = np.random.uniform(-1, 1, n_total)
    inside = x ** 2 + y ** 2 <= 1
    pi_estimate = 4 * np.sum(inside) / n_total

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(x[inside], y[inside], c="steelblue", s=2, alpha=0.5, label="Inside circle")
    ax.scatter(x[~inside], y[~inside], c="salmon", s=2, alpha=0.5, label="Outside circle")
    circle = Circle((0, 0), 1, fill=False, color="black", linewidth=2)
    ax.add_patch(circle)
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.set_title(f"{n_total} darts -> pi ~ {pi_estimate:.4f} (true pi = {np.pi:.4f})")
    ax.legend(loc="upper right", fontsize=8)
    plt.show()

    print(f"Estimate: {pi_estimate:.4f}  |  Error: {abs(pi_estimate - np.pi):.4f}")

interact(explore,
         n_total=IntSlider(value=5000, min=10, max=20000, step=500),
         seed=IntSlider(value=42, min=0, max=100, step=1));

## Questions to think about

1. Drop n_total to 10. How far off is the estimate, and does it change a lot if you try a few different seed values while keeping n_total at 10? What does this tell you about small samples?
2. Raise n_total to 20000. Does the estimate get consistently closer to pi, or does it still move around with different seeds? Compare the typical error at n_total=5000 with the typical error at n_total=20000 across a few seeds.
3. The Concept note compares this simulation to how the brain builds "precise perception from noisy neural signals." Based on what you saw with the seed slider, is any single dart individually informative about pi? What does that imply about how the brain might combine many individually unreliable signals into a stable percept?
4. This notebook is linked to both Lab 01 (the blind spot, which fills in missing information) and Lab 07 (Troxler fading, which is about populations of adapting neurons). Which is the closer analogy for what this simulation demonstrates -- "filling in a gap with a guess" or "averaging out noise across many independent samples"? Justify your answer.

## Challenge

Extend the simulation to show how the *typical size* of the error shrinks as the number of
darts grows, by repeating the estimate many times at each sample size. This is the same
square-root-of-n law that governs how pooling many noisy neurons (as in Lab 13's
population-vector model) reduces uncertainty.

In [ ]:
ns_to_test = [10, 100, 1000, 10000]
n_repeats = 30
mean_abs_errors = []

for n in ns_to_test:
    errors = []
    for rep in range(n_repeats):
        x = np.random.uniform(-1, 1, n)
        y = np.random.uniform(-1, 1, n)
        inside = x ** 2 + y ** 2 <= 1
        pi_estimate = 4 * np.sum(inside) / n
        errors.append(abs(pi_estimate - np.pi))
    mean_abs_errors.append(np.mean(errors))

plt.figure(figsize=(6, 4))
plt.loglog(ns_to_test, mean_abs_errors, "o-", color="steelblue", label="Average error (30 repeats)")
plt.loglog(ns_to_test, [1 / np.sqrt(n) for n in ns_to_test], "k--", label="1 / sqrt(n) reference")
plt.xlabel("Number of darts (n)")
plt.ylabel("Average absolute error")
plt.legend()
plt.title("Error shrinks roughly as 1 / sqrt(n)")
plt.show()

## Record what you ran

In [ ]:
NOTEBOOK_ID = "appendix-1.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")